
# Лабораторна робота №2 — Частина 2

### Аналіз датасету Individual Household Electric Power Consumption Dataset за допомогою pandas DataFrame



## Іморт бібліотек

In [31]:
import pandas as pd
import numpy as np
import time
import timeit
import sys
from scipy import stats
from sklearn.preprocessing import StandardScaler, MinMaxScaler

## Завантаження датасету

In [8]:
df = pd.read_csv('household_power_consumption.txt', 
                 sep=';', 
                 na_values=['?'], 
                 low_memory=False)

df['dt'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], dayfirst=True)
df.set_index('dt', inplace=True)
df.drop(['Date', 'Time'], axis=1, inplace=True)

df.head()

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
dt,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


## Data Cleaning (Очищення даних)

In [15]:
def clean_data(data):
    # Також важливо переконатися, що типи даних числові для аналізу
    # оскільки після read_csv вони можуть бути об'єктами
    return data.dropna()

execution_time = timeit.timeit(lambda: clean_data(df), number=1)
df = clean_data(df)

print(f"Час виконання: {execution_time:.5f} секунд")
df.head()

Час виконання: 0.03119 секунд


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
dt,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


## Записи, у яких загальна активна споживана потужність перевищує 5 кВт

In [16]:
def filter_power(data):
    return data[data['Global_active_power'] > 5]

execution_time = timeit.timeit(lambda: filter_power(df), number=1)
result_1 = filter_power(df)

print(f"Час виконання: {execution_time:.5f} секунд")
print(f"Знайдено записів: {len(result_1)}")
result_1.head()

Час виконання: 0.00506 секунд
Знайдено записів: 17547


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
dt,,,,,,,
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0
2006-12-16 17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0


## Сила струму 19-20 А, де Sub_metering_2 > Sub_metering_3

In [17]:
def filter_current_and_appliances(data):
    return data[(data['Global_intensity'] >= 19) & 
                (data['Global_intensity'] <= 20) & 
                (data['Sub_metering_2'] > data['Sub_metering_3'])]

execution_time = timeit.timeit(lambda: filter_current_and_appliances(df), number=1)
result_2 = filter_current_and_appliances(df)

print(f"Час виконання: {execution_time:.5f} секунд")
print(f"Знайдено записів: {len(result_2)}")
result_2.head()

Час виконання: 0.01004 секунд
Знайдено записів: 2509


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
dt,,,,,,,
2006-12-16 18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
2006-12-17 01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
2006-12-17 01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0
2006-12-17 01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0
2006-12-17 01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0


## Випадкові 500,000 записів та середні величини груп споживанн

In [18]:
def random_sample_means(data):
    sample = data.sample(n=500000, replace=False)
    means = sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
    return means

execution_time = timeit.timeit(lambda: random_sample_means(df), number=1)
means = random_sample_means(df)

print(f"Час виконання: {execution_time:.5f} секунд")
print("Середні значення:")
print(means)

Час виконання: 0.13076 секунд
Середні значення:
Sub_metering_1    1.138754
Sub_metering_2    1.304168
Sub_metering_3    6.461760
dtype: float64


 ## Після 18:00, > 6 кВт, група 2 найбільша, кожен 3-й з першої і 4-й з другої половини

In [19]:
def complex_filter(data):
    target_time = data.between_time('18:00', '23:59')
    high_power = target_time[target_time['Global_active_power'] > 6]

    group2_max = high_power[(high_power['Sub_metering_2'] > high_power['Sub_metering_1']) & 
                            (high_power['Sub_metering_2'] > high_power['Sub_metering_3'])]
    n = len(group2_max)
    first_half = group2_max.iloc[:n//2].iloc[::3]
    second_half = group2_max.iloc[n//2:].iloc[::4]
    
    return pd.concat([first_half, second_half])

execution_time = timeit.timeit(lambda: complex_filter(df), number=1)
result_3 = complex_filter(df)

print(f"Час виконання: {execution_time:.5f} секунд")
print(f"Кількість відібраних записів: {len(result_3)}")
result_3.head()

Час виконання: 0.03948 секунд
Кількість відібраних записів: 310


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
dt,,,,,,,
2006-12-16 18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0
2006-12-16 18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0
2006-12-28 20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0
2006-12-28 21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0
2006-12-28 21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0


## Нормалізація 

In [22]:
numeric_cols = df.columns.tolist() 
def normalize_dataset(dataframe, columns):
    scaler = MinMaxScaler()
    normalized_df = dataframe.copy()
    normalized_df[columns] = scaler.fit_transform(normalized_df[columns])
    return normalized_df
execution_time = timeit.timeit(lambda: normalize_dataset(df, numeric_cols), number=1)
normalized_df = normalize_dataset(df, numeric_cols)

print(f"Час виконання: {execution_time:.5f} секунд")
print("Нормалізовані дані:")
normalized_df[numeric_cols].head()

Час виконання: 0.31292 секунд
Нормалізовані дані:


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
dt,,,,,,,
2006-12-16 17:24:00,0.374796,0.300719,0.376090,0.377593,0.0,0.0125,0.548387
2006-12-16 17:25:00,0.478363,0.313669,0.336995,0.473029,0.0,0.0125,0.516129
2006-12-16 17:26:00,0.479631,0.358273,0.326010,0.473029,0.0,0.0250,0.548387
2006-12-16 17:27:00,0.480898,0.361151,0.340549,0.473029,0.0,0.0125,0.548387
2006-12-16 17:28:00,0.325005,0.379856,0.403231,0.323651,0.0,0.0125,0.548387


## Стандартизація

In [24]:
scaler_std = StandardScaler()
df_standardized = df.copy()
def standardize_data(data, cols):
    data_copy = data.copy()
    data_copy[cols] = scaler_std.fit_transform(data[cols])
    return data_copy
execution_time = timeit.timeit(lambda: standardize_data(df, numeric_cols), number=1)
df_standardized = standardize_data(df, numeric_cols)

print(f"Час виконання стандартизації: {execution_time:.5f} секунд")
print("-" * 30)
print("Стандартизований датасет:")
df_standardized[numeric_cols].head()

Час виконання стандартизації: 0.46185 секунд
------------------------------
Стандартизований датасет:


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
dt,,,,,,,
2006-12-16 17:24:00,2.955077,2.610721,-1.851816,3.098789,-0.182337,-0.051274,1.249421
2006-12-16 17:25:00,4.037085,2.770406,-2.225274,4.133800,-0.182337,-0.051274,1.130897
2006-12-16 17:26:00,4.050326,3.320432,-2.330213,4.133800,-0.182337,0.120487,1.249421
2006-12-16 17:27:00,4.063567,3.355917,-2.191324,4.133800,-0.182337,-0.051274,1.249421
2006-12-16 17:28:00,2.434881,3.586573,-1.592556,2.513782,-0.182337,-0.051274,1.249421


## Коефіцієнт Пірсона та Спірмена

In [32]:
def calculate_correlations(data):
    col1 = 'Global_active_power'
    col2 = 'Global_intensity'

    pearson_r, pearson_p = stats.pearsonr(data[col1], data[col2])
    spearman_r, spearman_p = stats.spearmanr(data[col1], data[col2])

    return pearson_r, pearson_p, spearman_r, spearman_p

execution_time = timeit.timeit(lambda: calculate_correlations(df), number=1)

pearson_r, pearson_p, spearman_r, spearman_p = calculate_correlations(df)

print(f"Час виконання: {execution_time:.5f} секунд")
print(f"Пірсон:  r = {pearson_r:.4f}, p-value = {pearson_p:.4e}")
print(f"Спірмен: r = {spearman_r:.4f}, p-value = {spearman_p:.4e}")

Час виконання: 0.45916 секунд
Пірсон:  r = 0.9989, p-value = 0.0000e+00
Спірмен: r = 0.9954, p-value = 0.0000e+00


## One Hot Encoding категоріального атрибута (година доби)

In [35]:
def perform_combined_encoding(data):
    # Працюємо з копією
    df_temp = data.copy()

    df_temp['hour_category'] = pd.cut(df_temp.index.hour, 
                                      bins=[0, 6, 12, 18, 24], 
                                      labels=['ніч', 'ранок', 'день', 'вечір'], 
                                      right=False)
    df_temp['day_type'] = np.where(df_temp.index.weekday < 5, 'Weekday', 'Weekend')
    df_encoded = pd.get_dummies(df_temp, columns=['hour_category', 'day_type'], dtype=int)
    
    return df_encoded

execution_time = timeit.timeit(lambda: perform_combined_encoding(df.head(100000)), number=1)
result_final = perform_combined_encoding(df.head(100000))

print(f"Час виконання: {execution_time:.5f} секунд")
print("-" * 30)
print("Результат One Hot Encoding (перші 10 рядків):")
new_cols = [col for col in result_final.columns if 'hour_category' in col or 'day_type' in col]
result_final[new_cols].head(10)

Час виконання: 0.04131 секунд
------------------------------
Результат One Hot Encoding (перші 10 рядків):


,hour_category_ніч,hour_category_ранок,hour_category_день,hour_category_вечір,day_type_Weekday,day_type_Weekend
dt,,,,,,
2006-12-16 17:24:00,0,0,1,0,0,1
2006-12-16 17:25:00,0,0,1,0,0,1
2006-12-16 17:26:00,0,0,1,0,0,1
2006-12-16 17:27:00,0,0,1,0,0,1
2006-12-16 17:28:00,0,0,1,0,0,1
2006-12-16 17:29:00,0,0,1,0,0,1
2006-12-16 17:30:00,0,0,1,0,0,1
2006-12-16 17:31:00,0,0,1,0,0,1
2006-12-16 17:32:00,0,0,1,0,0,1
